In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np

# Dosya yolunu kendi Drive'ındaki klasöre göre güncelle
file_path = '/content/drive/MyDrive/F1_Project/df_model_ready.parquet'

# Veriyi okuma
df = pd.read_parquet(file_path)

In [ ]:
# İlk 5 satıra bakalım
display(df.head())

# Sütun isimleri ve veri tiplerini görelim
print(df.info())

,raceId,driverId,constructorId,driver_name,constructor_name,year,round,circuit_name,date,season_progress,...,drv_pre_points,drv_pre_position,drv_pre_wins,con_pre_points,con_pre_position,con_pre_wins,gap_to_fastest_ms,driver_age,is_podium,personal_best_lap_ms
0,18,1,1,Lewis Hamilton,McLaren,2008,1,Albert Park Grand Prix Circuit,2008-03-16,0.055556,...,109.0,2.0,4.0,218.0,11.0,8.0,34.0,23.186858,1,87452.0
1,18,2,2,Nick Heidfeld,BMW Sauber,2008,1,Albert Park Grand Prix Circuit,2008-03-16,0.055556,...,61.0,5.0,0.0,101.0,2.0,0.0,321.0,30.850103,1,87739.0
2,18,3,3,Nico Rosberg,Williams,2008,1,Albert Park Grand Prix Circuit,2008-03-16,0.055556,...,20.0,9.0,0.0,33.0,4.0,0.0,672.0,22.718686,1,88090.0
3,18,4,4,Fernando Alonso,Renault,2008,1,Albert Park Grand Prix Circuit,2008-03-16,0.055556,...,109.0,3.0,4.0,51.0,3.0,0.0,1185.0,26.631075,0,88603.0
4,18,5,1,Heikki Kovalainen,McLaren,2008,1,Albert Park Grand Prix Circuit,2008-03-16,0.055556,...,30.0,7.0,0.0,218.0,11.0,8.0,0.0,26.406571,0,87418.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8970 entries, 0 to 8969
Data columns (total 31 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   raceId                8970 non-null   int64         
 1   driverId              8970 non-null   int64         
 2   constructorId         8970 non-null   int64         
 3   driver_name           8970 non-null   object        
 4   constructor_name      8970 non-null   object        
 5   year                  8970 non-null   int64         
 6   round                 8970 non-null   int64         
 7   circuit_name          8970 non-null   object        
 8   date                  8970 non-null   datetime64[us]
 9   season_progress       8970 non-null   float64       
 10  circuitId             8970 non-null   int64         
 11  lat                   8970 non-null   float64       
 12  lng                   8970 non-null   float64       
 13  alt               

In [ ]:
# 1. Önce veriyi kesinlikle tarihe göre sıralamalıyız. Zaman serisi mantığında bu şarttır.
df = df.sort_values('date')

# 2. Pilot bazında gruplama yapıp, son 3 yarıştaki ortalama sıralama pozisyonunu bulalım.
# shift(1) ile o anki yarışı hesaba katmıyoruz.
# min_periods=1 ile son 3 yarış yoksa bile (örn: sezonun 2. yarışı) elimizdeki kadarının ortalamasını alıyoruz.
df['recent_quali_form'] = df.groupby('driverId')['quali_position'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).mean()
)

# 3. Pilotun kariyerindeki ilk yarışında geçmiş verisi olmadığı için NaN (boş) dönecektir.
# Bu NaN değerleri, o pilotun o günkü güncel quali_position değeri ile dolduralım.
df['recent_quali_form'] = df['recent_quali_form'].fillna(df['quali_position'])

# 4. Yaptığımız işlemin doğruluğunu kontrol etmek için rastgele bir pilotun verilerine bakalım.
# Fernando Alonso'nun (driverId'si 4'tür genelde ama isme göre filtreleyelim) sonuçlarını görelim:
alonso_df = df[df['driver_name'].str.contains('Alonso', case=False, na=False)]
display(alonso_df[['driver_name', 'year', 'round', 'quali_position', 'recent_quali_form']].head(10))

,driver_name,year,round,quali_position,recent_quali_form
1880,Fernando Alonso,2003,1,10.0,10.000000
1896,Fernando Alonso,2003,2,1.0,10.000000
1916,Fernando Alonso,2003,3,10.0,5.500000
1939,Fernando Alonso,2003,4,8.0,7.000000
1955,Fernando Alonso,2003,5,3.0,6.333333
1989,Fernando Alonso,2003,6,19.0,7.000000
1998,Fernando Alonso,2003,7,8.0,10.000000
2017,Fernando Alonso,2003,8,4.0,10.000000
2037,Fernando Alonso,2003,9,8.0,10.333333
2071,Fernando Alonso,2003,10,7.0,6.666667


In [5]:
import numpy as np

# 1. Verinin tarihe göre sıralı olduğundan emin olalım (Kümülatif işlemler için şart)
df = df.sort_values('date')

# 2. Pilotun bu pistteki geçmiş yarış sayısı (Deneyim)
# cumcount() ilk yarış için 0, ikinci yarış için 1, üçüncü için 2 döndürür. (Tam istediğimiz sızıntısız hesaplama!)
df['circuit_experience'] = df.groupby(['driverId', 'circuitId']).cumcount()

# 3. Pilotun bu pistteki geçmiş podyum sayısı
# cumsum() o anki yarışın sonucunu da dahil eder. Sızıntıyı önlemek için o anki 'is_podium' değerini sonuçtan çıkarıyoruz.
df['circuit_podium_count'] = df.groupby(['driverId', 'circuitId'])['is_podium'].cumsum() - df['is_podium']

# 4. Pist Dominasyon Oranı = (Geçmiş Podyum Sayısı / Geçmiş Yarış Sayısı)
# Eğer pilot o pistte ilk defa yarışıyorsa (deneyim = 0), sıfıra bölünme hatası almamak için np.where kullanıyoruz ve oranı 0.0 yapıyoruz.
df['circuit_dominance_rate'] = np.where(
    df['circuit_experience'] > 0,
    df['circuit_podium_count'] / df['circuit_experience'],
    0.0
)

# 5. Kontrol edelim! Lewis Hamilton'ın Silverstone pistindeki verilerine bakalım:
hamilton_silverstone = df[
    (df['driver_name'].str.contains('Hamilton', case=False, na=False)) &
    (df['circuit_name'].str.contains('Silverstone', case=False, na=False))
]

display(hamilton_silverstone[['year', 'circuit_name', 'is_podium', 'circuit_experience', 'circuit_podium_count', 'circuit_dominance_rate']].head(10))

,year,circuit_name,is_podium,circuit_experience,circuit_podium_count,circuit_dominance_rate
546,2007,Silverstone Circuit,1,0,0,0.000000
168,2008,Silverstone Circuit,1,1,1,1.000000
2349,2009,Silverstone Circuit,0,2,2,1.000000
2751,2010,Silverstone Circuit,1,3,2,0.666667
3182,2011,Silverstone Circuit,0,4,3,0.750000
3642,2012,Silverstone Circuit,0,5,3,0.600000
4080,2013,Silverstone Circuit,0,6,3,0.500000
4517,2014,Silverstone Circuit,1,7,3,0.428571
4906,2015,Silverstone Circuit,1,8,4,0.500000
5324,2016,Silverstone Circuit,1,9,5,0.555556


In [6]:
# 1. Verinin tarihe göre sıralı olduğundan emin olalım
df = df.sort_values('date')

# --- ÖZELLİK 1: KARİYER PODYUM ORANI (Genel İstikrar) ---
# Pilotun o güne kadarki toplam yarıştığı yarış sayısı
df['career_experience'] = df.groupby('driverId').cumcount()

# Pilotun o güne kadarki toplam podyum sayısı (O anki yarışı çıkararak sızıntıyı önlüyoruz)
df['career_total_podiums'] = df.groupby('driverId')['is_podium'].cumsum() - df['is_podium']

# Kariyer podyum oranı (Sıfıra bölünme hatasını np.where ile engelliyoruz)
df['career_podium_rate'] = np.where(
    df['career_experience'] > 0,
    df['career_total_podiums'] / df['career_experience'],
    0.0
)

# --- ÖZELLİK 2: YAKIN DÖNEM PODYUM SERİSİ (Sıcak Form) ---
# Son 3 yarışta kaç kez podyuma çıktığını hesaplıyoruz.
# shift(1) ile o anki yarışı siliyoruz, rolling(window=3).sum() ile son 3'ü topluyoruz.
df['recent_podium_streak'] = df.groupby('driverId')['is_podium'].transform(
    lambda x: x.shift(1).rolling(window=3, min_periods=1).sum()
)

# İlk yarışlardaki NaN değerlerini 0 ile dolduruyoruz
df['recent_podium_streak'] = df['recent_podium_streak'].fillna(0)

# 3. Kontrol Edelim!
# Max Verstappen'in ortalığı kasıp kavurduğu 2023 sezonundaki serisine bakalım:
verstappen_df = df[
    (df['driver_name'].str.contains('Verstappen', case=False, na=False)) &
    (df['year'] == 2023)
]

display(verstappen_df[['round', 'circuit_name', 'is_podium', 'recent_podium_streak', 'career_podium_rate']].head(10))

,round,circuit_name,is_podium,recent_podium_streak,career_podium_rate
8051,1,Bahrain International Circuit,1,2.0,0.472393
8072,2,Jeddah Corniche Circuit,1,2.0,0.475610
8091,3,Albert Park Grand Prix Circuit,1,3.0,0.478788
8112,4,Baku City Circuit,1,3.0,0.481928
8131,5,Miami International Autodrome,1,3.0,0.485030
8151,6,Circuit de Monaco,1,3.0,0.488095
8171,7,Circuit de Barcelona-Catalunya,1,3.0,0.491124
8191,8,Circuit Gilles Villeneuve,1,3.0,0.494118
8211,9,Red Bull Ring,1,3.0,0.497076
8231,10,Silverstone Circuit,1,3.0,0.500000


In [7]:
# 1. Sadece hesaplama yapmak için kullandığımız, modelin kafasını karıştırabilecek
# veya sızıntı yapabilecek yardımcı sütunları (varsa) temizlemek iyi bir pratiktir.
# Ancak şu an ürettiğimiz tüm oranlar sızıntısız olduğu için veriyi doğrudan kaydedebiliriz.

# 2. Dosyayı yeni bir isimle Drive'a kaydediyoruz.
# Böylece orijinal dosya (df_model_ready.parquet) bozulmadan kalır.
save_path = '/content/drive/MyDrive/F1_Project/df_model_ready.parquet'
df.to_parquet(save_path)

print(f"Harika! Zenginleştirilmiş veri seti başarıyla kaydedildi:\n{save_path}")

Harika! Zenginleştirilmiş veri seti başarıyla kaydedildi:
/content/drive/MyDrive/F1_Project/df_model_ready.parquet


In [8]:
# Doğru yol ve isimle yeniden kaydediyoruz
correct_path = '/content/drive/MyDrive/F1_Project/df_train_ready.parquet'
df.to_parquet(correct_path)

print("Harika! Zenginleştirilmiş veri seti şimdi doğru isimle kaydedildi.")

Harika! Zenginleştirilmiş veri seti şimdi doğru isimle kaydedildi.


In [9]:
# Veri setinin genel yapısını ve toplam sütun sayısını kontrol edelim
print(df.info())

print("-" * 50)

# Sadece bizim sonradan eklediğimiz 8 sütunu (en sağdaki sütunları) ve ilk 5 satırı görelim
display(df.iloc[:, -8:].head())

<class 'pandas.core.frame.DataFrame'>
Index: 8970 entries, 1884 to 8969
Data columns (total 39 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   raceId                  8970 non-null   int64         
 1   driverId                8970 non-null   int64         
 2   constructorId           8970 non-null   int64         
 3   driver_name             8970 non-null   object        
 4   constructor_name        8970 non-null   object        
 5   year                    8970 non-null   int64         
 6   round                   8970 non-null   int64         
 7   circuit_name            8970 non-null   object        
 8   date                    8970 non-null   datetime64[us]
 9   season_progress         8970 non-null   float64       
 10  circuitId               8970 non-null   int64         
 11  lat                     8970 non-null   float64       
 12  lng                     8970 non-null   float64   

,recent_quali_form,circuit_experience,circuit_podium_count,circuit_dominance_rate,career_experience,career_total_podiums,career_podium_rate,recent_podium_streak
1884,20.0,0,0,0.0,0,0,0.0,0.0
1889,19.0,0,0,0.0,0,0,0.0,0.0
1886,18.0,0,0,0.0,0,0,0.0,0.0
1887,5.0,0,0,0.0,0,0,0.0,0.0
1893,2.0,0,0,0.0,0,0,0.0,0.0


In [10]:
# Dosyayı Drive'dan sıfırdan, yepyeni bir isimle okuyalım
# (Ahsen ve Yiğit'in de yapacağı işlem tam olarak budur)
df_kontrol = pd.read_parquet('/content/drive/MyDrive/F1_Project/df_train_ready.parquet')

# Sütunlarına bakalım, 39 sütun dönüyorsa dosyamız diskte de sapasağlam duruyor demektir!
print(df_kontrol.info())

<class 'pandas.core.frame.DataFrame'>
Index: 8970 entries, 1884 to 8969
Data columns (total 39 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   raceId                  8970 non-null   int64         
 1   driverId                8970 non-null   int64         
 2   constructorId           8970 non-null   int64         
 3   driver_name             8970 non-null   object        
 4   constructor_name        8970 non-null   object        
 5   year                    8970 non-null   int64         
 6   round                   8970 non-null   int64         
 7   circuit_name            8970 non-null   object        
 8   date                    8970 non-null   datetime64[us]
 9   season_progress         8970 non-null   float64       
 10  circuitId               8970 non-null   int64         
 11  lat                     8970 non-null   float64       
 12  lng                     8970 non-null   float64   